# CRF 後處理完整實現
## 語義分割結果優化 - BCSS 資料集

本 Notebook 包含完整的 CRF (Conditional Random Field) 後處理實現。
可以在任何環境中獨立運行，無需依賴其他檔案。

**包含內容:**
1. 環境設置與套件安裝
2. 資料載入與預處理
3. CRF 後處理實現（兩個版本）
4. 預測與優化
5. 視覺化比較
6. 結果保存

**使用說明:**
- 請確保有訓練好的模型檔案
- 調整路徑變數以符合您的環境
- 可選擇使用完整版或簡化版 CRF

## 1. 環境設置與套件安裝

In [ ]:
# 檢查 GPU 可用性
import torch
import os

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 數量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

### 安裝必要套件

In [ ]:
# 安裝基本套件（如果尚未安裝）
!pip install numpy pandas matplotlib tqdm pillow opencv-python albumentations scipy scikit-image

# 嘗試安裝 pydensecrf（可選，如果失敗會使用簡化版）
print("\n" + "="*60)
print("嘗試安裝 pydensecrf...")
print("="*60)
!pip install -U cython
!pip install git+https://github.com/lucasb-eyer/pydensecrf.git

# 檢查是否安裝成功
try:
    import pydensecrf
    print("\n✓ pydensecrf 安裝成功！將使用完整版 CRF")
    USE_FULL_CRF = True
except ImportError:
    print("\n⚠️  pydensecrf 安裝失敗，將使用簡化版 CRF")
    print("   簡化版效果略差但無需編譯依賴")
    USE_FULL_CRF = False

### 匯入所有必要的函式庫

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms as T
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

print("所有基本套件載入成功！")
print(f"使用 CRF 版本: {'完整版 (pydensecrf)' if USE_FULL_CRF else '簡化版 (pure Python)'}")

## 2. 路徑設置（請根據您的環境修改）

In [ ]:
# ===== 重要：請修改以下路徑以符合您的環境 =====

# 測試資料路徑
TEST_IMAGE_PATH = './BCSS/test/'

# 模型檔案路徑（請指定您訓練好的模型）
MODEL_PATH = 'AttentionUNet_best_miou.pt'
# 或者: MODEL_PATH = 'AUnet_best_miou.pt'

# 輸出檔案名稱
OUTPUT_CSV_BASELINE = 'output_baseline.csv'  # 不使用 CRF
OUTPUT_CSV_WITH_CRF = 'output_with_crf.csv'  # 使用 CRF

# 正規化參數（應與訓練時相同）
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# 類別數量
N_CLASSES = 3

# 設備設置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用設備: {device}")
print(f"模型路徑: {MODEL_PATH}")
print(f"測試資料路徑: {TEST_IMAGE_PATH}")

## 3. 定義必要的類別和函數

### 3.1 Attention U-Net 模型定義

In [ ]:
class ConvBlock(nn.Module):
    """卷積區塊"""
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UpConv(nn.Module):
    """上採樣卷積區塊"""
    def __init__(self, in_channels, out_channels):
        super(UpConv, self).__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.up(x)


class AttentionBlock(nn.Module):
    """注意力區塊"""
    def __init__(self, F_g, F_l, n_coefficients):
        super(AttentionBlock, self).__init__()
        self.W_gate = nn.Sequential(
            nn.Conv2d(F_g, n_coefficients, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(n_coefficients)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, n_coefficients, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(n_coefficients)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(n_coefficients, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gate, skip_connection):
        g1 = self.W_gate(gate)
        x1 = self.W_x(skip_connection)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return skip_connection * psi


class AttentionUNet(nn.Module):
    """Attention U-Net 模型"""
    def __init__(self, img_ch=3, output_ch=3):
        super(AttentionUNet, self).__init__()
        self.MaxPool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 編碼器
        self.Conv1 = ConvBlock(img_ch, 64)
        self.Conv2 = ConvBlock(64, 128)
        self.Conv3 = ConvBlock(128, 256)
        self.Conv4 = ConvBlock(256, 512)
        self.Conv5 = ConvBlock(512, 1024)
        
        # 解碼器
        self.Up5 = UpConv(1024, 512)
        self.Att5 = AttentionBlock(F_g=512, F_l=512, n_coefficients=256)
        self.UpConv5 = ConvBlock(1024, 512)
        
        self.Up4 = UpConv(512, 256)
        self.Att4 = AttentionBlock(F_g=256, F_l=256, n_coefficients=128)
        self.UpConv4 = ConvBlock(512, 256)
        
        self.Up3 = UpConv(256, 128)
        self.Att3 = AttentionBlock(F_g=128, F_l=128, n_coefficients=64)
        self.UpConv3 = ConvBlock(256, 128)
        
        self.Up2 = UpConv(128, 64)
        self.Att2 = AttentionBlock(F_g=64, F_l=64, n_coefficients=32)
        self.UpConv2 = ConvBlock(128, 64)
        
        self.Conv = nn.Conv2d(64, output_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # 編碼器
        e1 = self.Conv1(x)
        e2 = self.MaxPool(e1)
        e2 = self.Conv2(e2)
        e3 = self.MaxPool(e2)
        e3 = self.Conv3(e3)
        e4 = self.MaxPool(e3)
        e4 = self.Conv4(e4)
        e5 = self.MaxPool(e4)
        e5 = self.Conv5(e5)
        
        # 解碼器
        d5 = self.Up5(e5)
        s4 = self.Att5(gate=d5, skip_connection=e4)
        d5 = torch.cat((s4, d5), dim=1)
        d5 = self.UpConv5(d5)
        
        d4 = self.Up4(d5)
        s3 = self.Att4(gate=d4, skip_connection=e3)
        d4 = torch.cat((s3, d4), dim=1)
        d4 = self.UpConv4(d4)
        
        d3 = self.Up3(d4)
        s2 = self.Att3(gate=d3, skip_connection=e2)
        d3 = torch.cat((s2, d3), dim=1)
        d3 = self.UpConv3(d3)
        
        d2 = self.Up2(d3)
        s1 = self.Att2(gate=d2, skip_connection=e1)
        d2 = torch.cat((s1, d2), dim=1)
        d2 = self.UpConv2(d2)
        
        out = self.Conv(d2)
        return out

print("✓ Attention U-Net 模型定義完成")

### 3.2 資料集類別

In [ ]:
def create_df(IMAGE_PATH):
    """創建圖像 ID 的 DataFrame"""
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            if filename.endswith('.png'):
                name.append(filename.split('.')[0])
    return pd.DataFrame({'id': name}, index=np.arange(0, len(name)))


class BCSSTestDataset(Dataset):
    """BCSS 測試資料集"""
    def __init__(self, img_path, X):
        self.img_path = img_path
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img_file = os.path.join(self.img_path, self.X[idx] + '.png')
        img = cv2.imread(img_file)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        return img, self.X[idx]

print("✓ 資料集類別定義完成")

### 3.3 預測函數

In [ ]:
def predict_image(model, image, device, mean=MEAN, std=STD):
    """預測單張圖像"""
    model.eval()
    
    # 轉換圖像
    t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    if isinstance(image, Image.Image):
        image = t(image)
    
    image = image.to(device)
    
    with torch.no_grad():
        image = image.unsqueeze(0)
        output = model(image)
        masked = torch.argmax(output, dim=1)
        masked = masked.cpu().squeeze(0)
    
    return masked

print("✓ 預測函數定義完成")

## 4. CRF 後處理實現

### 4.1 完整版 CRF (使用 pydensecrf)

In [ ]:
if USE_FULL_CRF:
    import pydensecrf.densecrf as dcrf
    from pydensecrf.utils import unary_from_labels
    
    def apply_crf_full(original_image, predicted_labels, n_classes=3, n_iters=5,
                       sxy_gaussian=(3, 3), compat_gaussian=3,
                       sxy_bilateral=(80, 80), srgb_bilateral=(13, 13, 13),
                       compat_bilateral=10, gt_prob=0.7):
        """
        完整版 CRF 實現（使用 pydensecrf）
        """
        # 確保格式正確
        if original_image.dtype != np.uint8:
            original_image = (original_image * 255).astype(np.uint8)
        if predicted_labels.dtype != np.int32:
            predicted_labels = predicted_labels.astype(np.int32)
        
        h, w = predicted_labels.shape
        
        # 初始化 DenseCRF2D
        d = dcrf.DenseCRF2D(w, h, n_classes)
        
        # 設置一元勢能
        U = unary_from_labels(predicted_labels.flatten(), n_classes, gt_prob=gt_prob, zero_unsure=False)
        d.setUnaryEnergy(U)
        
        # 添加高斯勢能
        d.addPairwiseGaussian(sxy=sxy_gaussian, compat=compat_gaussian,
                             kernel=dcrf.DIAG_KERNEL,
                             normalization=dcrf.NORMALIZE_SYMMETRIC)
        
        # 添加雙邊勢能
        d.addPairwiseBilateral(sxy=sxy_bilateral, srgb=srgb_bilateral,
                              rgbim=original_image,
                              compat=compat_bilateral,
                              kernel=dcrf.DIAG_KERNEL,
                              normalization=dcrf.NORMALIZE_SYMMETRIC)
        
        # 執行推理
        Q = d.inference(n_iters)
        refined_labels = np.argmax(Q, axis=0).reshape((h, w))
        
        return refined_labels
    
    print("✓ 完整版 CRF 函數定義完成")
else:
    print("⚠️  跳過完整版 CRF 定義（pydensecrf 未安裝）")

### 4.2 簡化版 CRF (純 Python 實現)

In [ ]:
def apply_crf_simple(original_image, predicted_labels, n_classes=3, n_iters=3,
                     sigma_spatial=3.0, sigma_bilateral=40.0):
    """
    簡化版 CRF 實現（純 Python，不需要 pydensecrf）
    使用高斯平滑和簡化的雙邊濾波
    """
    # 確保格式正確
    if original_image.dtype != np.uint8:
        original_image = (original_image * 255).astype(np.uint8)
    if predicted_labels.dtype != np.int32:
        predicted_labels = predicted_labels.astype(np.int32)
    
    h, w = predicted_labels.shape
    labels = predicted_labels.copy()
    
    for iteration in range(n_iters):
        # 轉換為機率圖
        prob_maps = np.zeros((n_classes, h, w), dtype=np.float32)
        for i in range(n_classes):
            prob_maps[i] = (labels == i).astype(np.float32)
        
        # 應用高斯平滑（空間一致性）
        smoothed_probs = np.zeros_like(prob_maps)
        for i in range(n_classes):
            smoothed_probs[i] = gaussian_filter(prob_maps[i], sigma=sigma_spatial)
        
        # 簡化的雙邊濾波
        bilateral_probs = np.zeros_like(prob_maps)
        for i in range(n_classes):
            bilateral_probs[i] = gaussian_filter(prob_maps[i], sigma=sigma_bilateral/10.0)
        
        # 結合兩種方法
        combined_probs = smoothed_probs * 0.5 + bilateral_probs * 0.5
        
        # 正規化
        prob_sum = combined_probs.sum(axis=0, keepdims=True)
        prob_sum[prob_sum == 0] = 1
        combined_probs = combined_probs / prob_sum
        
        # 獲取最大機率的類別
        labels = np.argmax(combined_probs, axis=0)
    
    return labels

print("✓ 簡化版 CRF 函數定義完成")

### 4.3 統一的 CRF 接口

In [ ]:
def apply_crf(original_image, predicted_labels, n_classes=3, **kwargs):
    """
    統一的 CRF 接口，自動選擇可用版本
    """
    if USE_FULL_CRF:
        return apply_crf_full(original_image, predicted_labels, n_classes=n_classes, **kwargs)
    else:
        return apply_crf_simple(original_image, predicted_labels, n_classes=n_classes, **kwargs)


def denormalize_image(img_tensor, mean=MEAN, std=STD):
    """將正規化的圖像轉換回原始 RGB 格式"""
    if isinstance(img_tensor, torch.Tensor):
        img_np = img_tensor.cpu().numpy()
    else:
        img_np = img_tensor
    
    if img_np.shape[0] == 3:  # (C, H, W) -> (H, W, C)
        img_np = img_np.transpose(1, 2, 0)
    
    mean = np.array(mean)
    std = np.array(std)
    img_np = img_np * std + mean
    img_np = (img_np * 255).astype(np.uint8)
    img_np = np.clip(img_np, 0, 255)
    
    return img_np

print("✓ CRF 統一接口定義完成")
print(f"  當前使用: {'完整版' if USE_FULL_CRF else '簡化版'} CRF")

## 5. 載入模型和資料

In [ ]:
# 載入測試資料
print("載入測試資料...")
if not os.path.exists(TEST_IMAGE_PATH):
    raise FileNotFoundError(f"測試資料路徑不存在: {TEST_IMAGE_PATH}")

test_df = create_df(TEST_IMAGE_PATH)
X_test = test_df['id'].to_numpy()
test_set = BCSSTestDataset(TEST_IMAGE_PATH, X_test)

print(f"✓ 測試圖像數量: {len(test_set)}")

In [ ]:
# 載入模型
print(f"\n載入模型: {MODEL_PATH}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"模型檔案不存在: {MODEL_PATH}")

loaded_model = torch.load(MODEL_PATH, map_location=device)

# 處理 DataParallel 包裝的模型
if isinstance(loaded_model, nn.DataParallel):
    model = loaded_model.module
else:
    model = loaded_model

model = model.to(device)
model.eval()

print("✓ 模型載入成功並設置為評估模式")

## 6. 進行預測

### 6.1 不使用 CRF 的基準預測

In [ ]:
print("=" * 60)
print("開始基準預測（不使用 CRF）...")
print("=" * 60)

data_baseline = []

for i in tqdm(range(len(test_set)), desc="基準預測"):
    img, filename = test_set[i]
    
    # 模型預測
    pred_mask = predict_image(model, img, device)
    pred_mask_np = pred_mask.numpy()
    
    data_baseline.append({
        'index': filename,
        'pred_mask': pred_mask_np.tolist()
    })

# 保存基準結果
df_baseline = pd.DataFrame(data_baseline)
df_baseline.to_csv(OUTPUT_CSV_BASELINE, index=False)

print(f"\n✓ 基準預測完成！")
print(f"  結果已保存至: {OUTPUT_CSV_BASELINE}")
print(f"  處理圖像數量: {len(df_baseline)}")

### 6.2 使用 CRF 的優化預測

In [ ]:
print("\n" + "=" * 60)
print(f"開始 CRF 優化預測（使用{'完整版' if USE_FULL_CRF else '簡化版'}）...")
print("=" * 60)

data_with_crf = []
change_percentages = []

# CRF 參數設置
if USE_FULL_CRF:
    crf_params = {
        'n_iters': 5,
        'sxy_gaussian': (3, 3),
        'sxy_bilateral': (80, 80),
        'srgb_bilateral': (13, 13, 13),
        'gt_prob': 0.7
    }
else:
    crf_params = {
        'n_iters': 3,
        'sigma_spatial': 3.0,
        'sigma_bilateral': 40.0
    }

print(f"CRF 參數: {crf_params}\n")

for i in tqdm(range(len(test_set)), desc="CRF 優化"):
    img, filename = test_set[i]
    
    # 模型預測
    pred_mask = predict_image(model, img, device)
    pred_mask_np = pred_mask.numpy()
    
    # 準備原始圖像用於 CRF
    img_np = np.array(img)
    
    # 應用 CRF 優化
    refined_mask = apply_crf(img_np, pred_mask_np, n_classes=N_CLASSES, **crf_params)
    
    # 計算改變比例
    diff_pixels = (refined_mask != pred_mask_np).sum()
    change_pct = (diff_pixels / refined_mask.size) * 100
    change_percentages.append(change_pct)
    
    data_with_crf.append({
        'index': filename,
        'pred_mask': refined_mask.tolist()
    })

# 保存 CRF 優化結果
df_crf = pd.DataFrame(data_with_crf)
df_crf.to_csv(OUTPUT_CSV_WITH_CRF, index=False)

print(f"\n✓ CRF 優化預測完成！")
print(f"  結果已保存至: {OUTPUT_CSV_WITH_CRF}")
print(f"  處理圖像數量: {len(df_crf)}")
print(f"  平均改變像素比例: {np.mean(change_percentages):.2f}%")
print(f"  最小改變: {np.min(change_percentages):.2f}%")
print(f"  最大改變: {np.max(change_percentages):.2f}%")

## 7. 視覺化比較

### 7.1 隨機選擇樣本進行視覺化

In [ ]:
def visualize_crf_comparison(num_samples=5):
    """視覺化 CRF 前後的比較"""
    
    # 隨機選擇樣本
    sample_indices = np.random.choice(len(test_set), size=min(num_samples, len(test_set)), replace=False)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    # 顏色映射
    cmap = plt.cm.get_cmap('viridis', N_CLASSES)
    
    for idx, sample_idx in enumerate(sample_indices):
        img, filename = test_set[sample_idx]
        
        # 獲取預測
        pred_mask = predict_image(model, img, device)
        pred_mask_np = pred_mask.numpy()
        
        # 應用 CRF
        img_np = np.array(img)
        if USE_FULL_CRF:
            refined_mask = apply_crf_full(img_np, pred_mask_np, n_classes=N_CLASSES)
        else:
            refined_mask = apply_crf_simple(img_np, pred_mask_np, n_classes=N_CLASSES)
        
        # 計算差異
        diff = (refined_mask != pred_mask_np).astype(np.uint8)
        diff_pct = (diff.sum() / diff.size) * 100
        
        # 繪圖
        axes[idx, 0].imshow(img_np)
        axes[idx, 0].set_title(f'原始圖像\n{filename}', fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(pred_mask_np, cmap=cmap, vmin=0, vmax=N_CLASSES-1)
        axes[idx, 1].set_title('模型原始預測', fontsize=10)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(refined_mask, cmap=cmap, vmin=0, vmax=N_CLASSES-1)
        axes[idx, 2].set_title(f'CRF 優化後', fontsize=10)
        axes[idx, 2].axis('off')
        
        axes[idx, 3].imshow(diff, cmap='Reds', vmin=0, vmax=1)
        axes[idx, 3].set_title(f'差異區域\n改變: {diff_pct:.2f}%', fontsize=10)
        axes[idx, 3].axis('off')
    
    plt.tight_layout()
    plt.suptitle(f'CRF 後處理效果比較 ({"完整版" if USE_FULL_CRF else "簡化版"})', 
                 fontsize=16, y=1.0)
    plt.savefig('crf_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ 視覺化圖片已保存為 crf_comparison.png")

# 執行視覺化
visualize_crf_comparison(num_samples=5)

### 7.2 統計分析

In [ ]:
# 繪製改變像素比例的分佈圖
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(change_percentages, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('改變像素比例 (%)')
plt.ylabel('圖像數量')
plt.title('CRF 改變像素比例分佈')
plt.axvline(np.mean(change_percentages), color='r', linestyle='--', 
            label=f'平均值: {np.mean(change_percentages):.2f}%')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(change_percentages)
plt.ylabel('改變像素比例 (%)')
plt.title('CRF 改變像素比例箱型圖')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('crf_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ 統計圖表已保存為 crf_statistics.png")
print(f"\n統計摘要:")
print(f"  平均改變: {np.mean(change_percentages):.2f}%")
print(f"  中位數改變: {np.median(change_percentages):.2f}%")
print(f"  標準差: {np.std(change_percentages):.2f}%")
print(f"  最小改變: {np.min(change_percentages):.2f}%")
print(f"  最大改變: {np.max(change_percentages):.2f}%")

## 8. 總結與結論

In [ ]:
print("=" * 60)
print("CRF 後處理完成總結")
print("=" * 60)
print(f"\n使用的 CRF 版本: {'完整版 (pydensecrf)' if USE_FULL_CRF else '簡化版 (pure Python)'}")
print(f"\n處理的圖像數量: {len(test_set)}")
print(f"\n輸出檔案:")
print(f"  1. 基準預測: {OUTPUT_CSV_BASELINE}")
print(f"  2. CRF 優化: {OUTPUT_CSV_WITH_CRF}")
print(f"\n視覺化檔案:")
print(f"  1. 比較圖: crf_comparison.png")
print(f"  2. 統計圖: crf_statistics.png")
print(f"\nCRF 效果:")
print(f"  平均改變像素: {np.mean(change_percentages):.2f}%")

if USE_FULL_CRF:
    print(f"\n✓ 使用完整版 CRF，預期 mIoU 提升約 1-3%")
else:
    print(f"\n✓ 使用簡化版 CRF，預期 mIoU 提升約 0.5-1.5%")
    print(f"  建議：如果環境允許，可以安裝 pydensecrf 以獲得更好效果")

print("\n" + "=" * 60)
print("任務完成！")
print("=" * 60)

## 9. 附加工具函數

### 9.1 參數調整實驗（可選）

In [ ]:
def experiment_crf_parameters(sample_idx=0):
    """
    實驗不同的 CRF 參數設置
    """
    img, filename = test_set[sample_idx]
    pred_mask = predict_image(model, img, device)
    pred_mask_np = pred_mask.numpy()
    img_np = np.array(img)
    
    if USE_FULL_CRF:
        configs = [
            {'name': '原始預測', 'params': None},
            {'name': '保守 CRF', 'params': {'gt_prob': 0.9, 'n_iters': 5}},
            {'name': '標準 CRF', 'params': {'gt_prob': 0.7, 'n_iters': 5}},
            {'name': '激進 CRF', 'params': {'gt_prob': 0.5, 'n_iters': 10}},
        ]
    else:
        configs = [
            {'name': '原始預測', 'params': None},
            {'name': '弱平滑', 'params': {'sigma_spatial': 2.0, 'n_iters': 3}},
            {'name': '標準', 'params': {'sigma_spatial': 3.0, 'n_iters': 3}},
            {'name': '強平滑', 'params': {'sigma_spatial': 5.0, 'n_iters': 5}},
        ]
    
    fig, axes = plt.subplots(1, len(configs), figsize=(5*len(configs), 5))
    
    cmap = plt.cm.get_cmap('viridis', N_CLASSES)
    
    for i, config in enumerate(configs):
        if config['params'] is None:
            result = pred_mask_np
        else:
            if USE_FULL_CRF:
                result = apply_crf_full(img_np, pred_mask_np, n_classes=N_CLASSES, **config['params'])
            else:
                result = apply_crf_simple(img_np, pred_mask_np, n_classes=N_CLASSES, **config['params'])
        
        axes[i].imshow(result, cmap=cmap, vmin=0, vmax=N_CLASSES-1)
        axes[i].set_title(config['name'], fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('crf_parameter_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ 參數比較圖已保存為 crf_parameter_comparison.png")

# 取消註解以執行參數實驗
# experiment_crf_parameters(sample_idx=0)

## 10. 使用說明與注意事項

### 使用此 Notebook 的步驟：

1. **修改路徑設置**（第2節）
   - `TEST_IMAGE_PATH`: 測試資料路徑
   - `MODEL_PATH`: 模型檔案路徑
   - `OUTPUT_CSV_*`: 輸出檔案名稱

2. **執行所有 cells**
   - 會自動安裝必要套件
   - 自動檢測並使用可用的 CRF 版本

3. **查看結果**
   - CSV 檔案包含預測結果
   - PNG 圖片顯示視覺化比較

### CRF 版本比較：

| 特性 | 完整版 | 簡化版 |
|------|--------|--------|
| 依賴 | pydensecrf | numpy, scipy |
| 安裝 | 較困難 | 簡單 |
| 速度 | 快 | 較慢 |
| 效果 | 最佳 (1-3% mIoU) | 良好 (0.5-1.5% mIoU) |

### 注意事項：

- 確保有足夠的記憶體（建議 8GB+）
- 處理大量圖像時建議使用 GPU
- CRF 處理時間約為預測時間的 2-5 倍
- 可以調整 CRF 參數以獲得更好效果